# Chapter 14 — Maximum Likelihood Estimation (MLE)
*Track 3: Data Scientists*

## 🎯 Learning Objectives
- Understand MLE as a principled way to fit distributions to data
- Derive MLE analytically for common distributions
- Implement numerical MLE with scipy.optimize

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import scipy.optimize as opt

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries loaded ✅")

## 1. Concept Review — What Is MLE?

**Maximum Likelihood Estimation** answers: *which parameter values make the
observed data most probable?*

$$\hat{\theta}_{MLE} = \arg\max_{\theta} \prod_{i=1}^n p(x_i | \theta)$$

In practice, maximise the **log-likelihood** (same maximum, numerically better):
$$\hat{\theta}_{MLE} = \arg\max_{\theta} \sum_{i=1}^n \log p(x_i | \theta)$$

In [ ]:
# Visualise likelihood as function of parameter
data = rng.normal(5, 2, 30)
mu_values = np.linspace(2, 8, 200)
log_likelihoods = [stats.norm.logpdf(data, mu, 2).sum() for mu in mu_values]

plt.plot(mu_values, log_likelihoods)
plt.axvline(data.mean(), color="red", linestyle="--", label=f"MLE μ={data.mean():.2f}")
plt.xlabel("μ"); plt.ylabel("Log-Likelihood")
plt.title("Log-Likelihood as a Function of μ (σ=2 known)")
plt.legend(); plt.tight_layout(); plt.show()

## 2. Math Walkthrough — Deriving MLE for Gaussian

For $X \sim N(\mu, \sigma^2)$:
$$\ell(\mu,\sigma) = -n\ln\sigma - \frac{1}{2\sigma^2}\sum(x_i - \mu)^2 + C$$

Setting $\frac{\partial \ell}{\partial \mu} = 0$:
$$\hat\mu_{MLE} = \frac{1}{n}\sum x_i = \bar x$$

Setting $\frac{\partial \ell}{\partial \sigma} = 0$:
$$\hat\sigma^2_{MLE} = \frac{1}{n}\sum(x_i - \bar x)^2$$

⚠️ Note: MLE variance is **biased** (divides by n, not n-1).

In [ ]:
data = rng.normal(5, 2, 1000)
mu_mle = data.mean()
sigma_mle = data.std(ddof=0)   # MLE (biased)
sigma_unb = data.std(ddof=1)   # Unbiased

print(f"True μ=5, σ=2")
print(f"MLE:      μ={mu_mle:.3f}, σ={sigma_mle:.3f}")
print(f"Unbiased: μ={mu_mle:.3f}, σ={sigma_unb:.3f}")
print(f"Bias in σ²: {sigma_unb**2 - sigma_mle**2:.4f}")

## 3. Simulation — MLE for Exponential Distribution

$X \sim \text{Exp}(\lambda)$:  $f(x) = \lambda e^{-\lambda x}$

Log-likelihood: $\ell(\lambda) = n\ln\lambda - \lambda \sum x_i$

MLE: $\hat\lambda = \frac{n}{\sum x_i} = \frac{1}{\bar x}$

In [ ]:
lam_true = 0.5
data_exp = rng.exponential(1/lam_true, 200)

lam_values = np.linspace(0.1, 1.5, 300)
ll_exp = [len(data_exp)*np.log(l) - l*data_exp.sum() for l in lam_values]

lam_mle = 1 / data_exp.mean()
plt.plot(lam_values, ll_exp, label="Log-likelihood")
plt.axvline(lam_mle, color="red", linestyle="--", label=f"MLE λ={lam_mle:.3f}")
plt.axvline(lam_true, color="green", linestyle=":", label=f"True λ={lam_true}")
plt.xlabel("λ"); plt.ylabel("Log-likelihood")
plt.title("MLE for Exponential Distribution")
plt.legend(); plt.tight_layout(); plt.show()

## 4. Visualisation — Numerical MLE with scipy

For distributions where the analytical solution is complex, use `scipy.optimize`.

In [ ]:
# Fit a Beta distribution numerically
data_beta = rng.beta(2, 5, 500)

def neg_ll_beta(params):
    a, b = params
    if a <= 0 or b <= 0:
        return np.inf
    return -stats.beta.logpdf(data_beta, a, b).sum()

result = opt.minimize(neg_ll_beta, x0=[1, 1], method="Nelder-Mead")
a_hat, b_hat = result.x

x = np.linspace(0.001, 0.999, 200)
plt.hist(data_beta, bins=30, density=True, alpha=0.5, label="Data")
plt.plot(x, stats.beta.pdf(x, a_hat, b_hat), "r-", lw=2,
         label=f"MLE fit: a={a_hat:.2f}, b={b_hat:.2f}")
plt.plot(x, stats.beta.pdf(x, 2, 5), "g--", lw=2, label="True: a=2, b=5")
plt.title("Numerical MLE — Beta Distribution")
plt.legend(); plt.tight_layout(); plt.show()

## 5. Real Dataset Exercise — Fitting User Session Durations

In [ ]:
# Simulate user session times (right-skewed, like real web analytics)
sessions = rng.lognormal(mean=np.log(120), sigma=0.8, size=1000)

# Fit candidates
dist_candidates = {
    "Exponential": stats.expon,
    "Log-Normal": stats.lognorm,
    "Gamma": stats.gamma,
    "Weibull": stats.weibull_min,
}
print("Distribution Fitting Results:")
print(f"{'Distribution':<15} {'Log-L':>10} {'AIC':>10}")
for name, dist in dist_candidates.items():
    params = dist.fit(sessions)
    ll = dist.logpdf(sessions, *params).sum()
    k = len(params)
    aic = 2*k - 2*ll
    print(f"{name:<15} {ll:>10.1f} {aic:>10.1f}")

## 6. Practice Problems

**P1.** You observe 7 coin flips: H H T H H T H. Derive the MLE for p (P(Heads)) analytically.

**P2.** A Poisson process produces counts [3, 5, 2, 8, 4]. Write code to find MLE λ both
analytically (formula) and numerically (`scipy.optimize.minimize`). Verify they agree.

**P3.** Why is $\hat\sigma^2_{MLE} = \frac{1}{n}\sum(x_i - \bar x)^2$ biased?
Compute $E[\hat\sigma^2_{MLE}]$ to show the bias.

In [ ]:
# P1
heads, flips = 5, 7
p_mle = heads / flips
print(f"P1: MLE p = {heads}/{flips} = {p_mle:.4f}")

# P2
counts = np.array([3, 5, 2, 8, 4])
lam_analytical = counts.mean()
neg_ll = lambda lam: -stats.poisson.logpmf(counts, lam[0]).sum() if lam[0]>0 else np.inf
res = opt.minimize(neg_ll, [3.0], method="Nelder-Mead")
print(f"P2: Analytical λ={lam_analytical:.2f}, Numerical λ={res.x[0]:.2f}")